# Descarga del Dataset

## Workaround: Por la estructura de las carpetas le agregamos primero la ruta base del codigo /src/unidad_1

In [10]:
import sys
import os
from pathlib import Path

# 1. Agregamos la ruta base de nuestro código fuente al sistema de Python
ruta_src = str(Path.cwd().parent / "src" / "unidad_1")
if ruta_src not in sys.path:
    sys.path.append(ruta_src)

# 2. Cambiamos el directorio de trabajo a la raíz del proyecto
if Path.cwd().name == "notebooks":
    os.chdir("..")
    
print("Directorio de trabajo actual:", Path.cwd())


Directorio de trabajo actual: C:\Users\Anibal\Desktop\proyectos\Maestria en AI UASD\Ciencia de Datos II\Tareas\Unidad 1


## Descargamos nuestro Dataset en carpeta data/raw/dataset.csv

In [11]:
from inf8239_u01.data import download_csv
# 3. Descargamos los datos
URL = "https://raw.githubusercontent.com/sistemasperez/-INF-8239-Unidad-1/refs/heads/main/data/ai_student_impact_dataset.csv"
path = download_csv(URL)
print("Archivo descargado en:", path)

Archivo descargado en: data\raw\dataset.csv


## Importamos las librerias

In [4]:
import pandas as pd

## Cargar y Reconocer el esquema de los datos

In [12]:
# Escogemos solo 5000 estudiantes para hacerlo en el CPU
# Quitar la función sample si lo queremos correr por el dataset completo
df = pd.read_csv("data/raw/dataset.csv").sample(n=5000, random_state=42)
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.tail())
assert not df.empty

(5000, 16)
Student_ID                      int64
Major_Category                 object
Year_of_Study                  object
Pre_Semester_GPA              float64
Weekly_GenAI_Hours            float64
Primary_Use_Case               object
Prompt_Engineering_Skill       object
Tool_Diversity                  int64
Paid_Subscription                bool
Traditional_Study_Hours       float64
Perceived_AI_Dependency         int64
Institutional_Policy           object
Anxiety_Level_During_Exams      int64
Post_Semester_GPA             float64
Skill_Retention_Score         float64
Burnout_Risk_Level             object
dtype: object
       Student_ID Major_Category Year_of_Study  Pre_Semester_GPA  \
33553      133554           STEM      Graduate             2.208   
9427       109428           Arts      Graduate             2.931   
199        100200           Arts     Sophomore             2.874   
12447      112448        Medical        Junior             2.517   
39489      139490       Bus

## Auditoria de los datos

In [6]:
audit = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "porcentaje_ausente": (df.isna().mean()*100).round(2),
    "unicos": df.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)
print("Duplicados:", df.duplicated().sum())
display(audit)

Duplicados: 0


,tipo,ausentes,porcentaje_ausente,unicos
Student_ID,int64,0,0.0,5000
Major_Category,object,0,0.0,5
Year_of_Study,object,0,0.0,5
Pre_Semester_GPA,float64,0,0.0,1724
Weekly_GenAI_Hours,float64,0,0.0,1935
Primary_Use_Case,object,0,0.0,5
Prompt_Engineering_Skill,object,0,0.0,3
Tool_Diversity,int64,0,0.0,5
Paid_Subscription,bool,0,0.0,2
Traditional_Study_Hours,float64,0,0.0,1812


In [ ]:
📁 AI_Student_Impact_Dataset
├── 🪪 Identifier          → Student_ID
├── 🎓 Academic Profile    → Major_Category, Year_of_Study, Pre/Post GPA
├── 🤖 AI Behaviour        → Weekly_GenAI_Hours, Primary_Use_Case,
│                            Prompt_Engineering_Skill, Tool_Diversity,
│                            Paid_Subscription
├── 📚 Study Habits        → Traditional_Study_Hours, Perceived_AI_Dependency
├── 🏛️ Institutional       → Institutional_Policy
└── 🧠 Well-being          → Anxiety_Level, Skill_Retention_Score,
                             Burnout_Risk_Level


## Definir target y retirar fugas

In [7]:
TARGET = "Burnout_Risk_Level"
DROP_COLUMNS = ["Student_ID", "Post_Semester_GPA", "Skill_Retention_Score"]  # deja [] si no aplica
assert TARGET in df.columns
X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]
print(y.value_counts(dropna=False))
assert y.notna().all()
assert y.nunique() >= 2

Burnout_Risk_Level
Medium    2077
Low       1655
High      1268
Name: count, dtype: int64


In [ ]:
# por que tomamos esta desición:
# Post_Semester_GPA fue eliminada por alta fuga de datos:
#Como nuestro objetivo es tomar una decisión anticipada (ayudar al estudiante antes de que colapse), 
# no podemos usar sus notas de final de semestre porque en el mundo real, en el momento de tomar la decisión, 
# esa nota aún no existe.

# Student_ID fue eliminado riesgo de sobreajuste. 
# Al ser un ID, no aporta valor predictivo y la IA podría intentar memorizar los números de los estudiantes en lugar de 
# aprender patrones reales.

# Skill_Retention_Score: Al igual que Post_Semester_GPA, esta puntuación solo se conoce cuando el semestre ya acabó. 

## Separar tipos y crear preprocesamiento

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
# Si falta un valor rellenalo con la media y usa un escalador estandar para 
# que los valores numéricos de las colummnas tengan una misma escala
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])

# Si falta un valor rellenalo con el mas frecuente (Moda)
# Usa  OneHotEncoder para que las categorias sean variables boleanas (0/1)
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocess = ColumnTransformer([("num", num_pipe, num_cols),
                                ("cat", cat_pipe, cat_cols)])
print(len(num_cols), len(cat_cols))

6 6


In [ ]:
# Transformaciones de variables para escalar o estandarizar
# Weekly_GenAI_Hours (Estandarización matemática (`StandardScaler`))
# Primary_Use_Case (Codificación binaria (`OneHotEncoder`)
# Perceived_AI_Dependency (Estandarización matemática (`StandardScaler`))
# Traditional_Study_Hours (Estandarización matemática (`StandardScaler`)


## Dividir, baseline y SVM

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

# Dividir las cartas de estudio y las del examen
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)

# La competencia entre un modelo "tonto" (Dummy) y tu verdadera Inteligencia Artificial (SVM) para
# medir si tu IA realmente está aprendiendo algo útil.
dummy=Pipeline([("prep",preprocess),("model",DummyClassifier(strategy="most_frequent"))])
svm=Pipeline([("prep",preprocess),("model",SVC(C=1,gamma="scale",probability=True,random_state=42))])

# Si los ponemos a competir y tu SVC saca un 55/100, significa que tu IA es peor que simplemente adivinar.
for name,model in {"dummy":dummy,"svm":svm}.items():
    model.fit(Xtr,ytr)
    pred=model.predict(Xte)
    print(name, f1_score(yte,pred,average="macro"))
print(classification_report(yte,svm.predict(Xte)))

dummy 0.19552414605418136
svm 0.4917372598197706
              precision    recall  f1-score   support

        High       0.63      0.47      0.54       254
         Low       0.47      0.44      0.45       331
      Medium       0.44      0.54      0.48       415

    accuracy                           0.49      1000
   macro avg       0.52      0.48      0.49      1000
weighted avg       0.50      0.49      0.49      1000

